# Functional Node Classification and Bridge Identification — Hypothesis 3

Classifies citizen organizations, public/private policy hubs, and recurring intermediate bridge nodes. Korean entity names are retained because they are source-data identifiers.


In [ ]:
import pandas as pd
import networkx as nx
from collections import Counter

# 1. Load nodes and initialize functional node types
nodes = pd.read_csv("nodes_2_1.csv")
nodes["normalized_label"] = nodes["Label"].str.strip().str.lower()
nodes["type"] = "others"

# 2. Remove generic labels that are not informative organization names
# Korean terms are retained because matching is performed against Korean source entities.
stopwords = [
    "한국", "자원", "공사", "센터", "정보", "기관", "통합", "기술", "지원", "유일", "미래산업",
    "산업", "협회", "연구", "중앙", "전국", "사회", "재단", "공동", "개발", "MOU", "CES", "IoT"
]
nodes = nodes[
    ~nodes["normalized_label"].isin(stopwords) &
    (nodes["normalized_label"].str.len() > 2)
]

# 3. Define aliases for public and private policy hubs
policy_aliases = {
    "국토교통부": ["국토교통부"], "행정안전부": ["행정안전부"],
    "과학기술정보통신부": ["과학기술정보통신부"], "보건복지부": ["보건복지부"],
    "환경부": ["환경부"], "서울특별시": ["서울특별시"], "부산광역시": ["부산광역시"],
    "세종특별자치시": ["세종특별자치시"], "경기도": ["경기도"],
    "한국도로공사": ["한국도로공사"], "한국환경공단": ["한국환경공단"],
    "K-water": ["k-water"], "한국지역정보개발원": ["한국지역정보개발원"],
    "KT": ["KT", "케이티", "케이티(KT)"], "SK": ["SK", "SKT", "SK텔레콤", "SK브로드밴드"],
    "삼성": ["삼성", "삼성SDS", "삼성전자"], "LG": ["LG", "LG전자", "엘지", "LG유플러스"],
    "LH": ["LH", "LH공사", "한국토지주택공사(LH)"], "IH": ["IH", "IH공사"],
    "카카오": ["카카오", "카카오모빌리티"], "네이버": ["네이버", "라인", "NAVER"],
    "현대": ["현대", "현대자동차"]
}
private_policy_labels = {"KT", "SK", "삼성", "LG", "LH", "IH", "카카오", "네이버", "현대"}

# 4. Assign policy-hub types using alias matching
for label, aliases in policy_aliases.items():
    type_value = "private_policy" if label in private_policy_labels else "policy"
    nodes.loc[
        nodes["normalized_label"].apply(
            lambda value: any(alias.lower() in value for alias in aliases)
        ),
        "type"
    ] = type_value

# 5. Identify citizen-participation organizations using keyword rules and a manual list
citizen_keywords = ["리빙랩", "npo", "복지재단", "센터", "마을", "협동조합", "ymca", "공동체"]
citizen_auto = nodes[
    nodes["normalized_label"].apply(
        lambda value: any(keyword in value for keyword in citizen_keywords)
    )
]["normalized_label"].tolist()

manual_citizen_nodes = [
    "서울혁신파크", "부산시 사회혁신센터", "청년허브", "서울특별시 마을공동체종합지원센터",
    "경기도공익활동지원센터", "서울시공익활동지원센터", "충청남도공익활동지원센터", "경상남도공익활동지원센터",
    "대구시 시민공익활동지원센터", "울산시공익활동지원센터", "천안 NGO 센터", "대전시 NGO 지원센터",
    "광주 NGO 시민재단", "충북시민사회지원센터", "부산시민운동지원센터",
    "경기시민사회연구소 울림", "부천희망재단", "사단법인 시민", "사단법인 공공", "천안시민사회네트워크",
    "충남시민재단", "충북시민재단", "지리산 이음", "지리산작은변화지원센터", "대구시민재단", "부산시민재단", "경북시민재단",
    "국민권익위원회", "국민신문고", "광화문1번가", "e-people", "정책브리핑", "행정안전부 주민참여예산",
    "세종시 열린시정", "광주 바로소통", "대전 소통ON", "희망제작소",
    "마을공동체종합지원센터", "서울시 마을공동체종합지원센터", "마을지원센터", "공동체지원센터"
]
manual_citizen_nodes = [name.strip().lower() for name in manual_citizen_nodes]
citizen_nodes = list(set(citizen_auto + manual_citizen_nodes))
nodes.loc[nodes["normalized_label"].isin(citizen_nodes), "type"] = "citizen"

# 6. Load edges and retain only edges whose endpoints remain in the node table
edges = pd.read_csv("edges_2_1.csv")
valid_ids = set(nodes["Id"])
edges = edges[
    edges["Source"].isin(valid_ids) & edges["Target"].isin(valid_ids)
]

# 7. Build the graph and identify recurring intermediate nodes on citizen-to-policy shortest paths
G = nx.from_pandas_edgelist(edges, "Source", "Target")
citizen_ids = nodes[nodes["type"] == "citizen"]["Id"].tolist()
policy_ids = nodes[nodes["type"].isin(["policy", "private_policy"])]["Id"].tolist()

bridge_candidates = []
valid_graph_nodes = set(G.nodes)
for citizen in citizen_ids:
    if citizen not in valid_graph_nodes:
        continue
    for policy in policy_ids:
        if policy not in valid_graph_nodes:
            continue
        try:
            path = nx.shortest_path(G, source=citizen, target=policy)
            bridge_candidates.extend(path[1:-1])
        except nx.NetworkXNoPath:
            continue

bridge_count = Counter(bridge_candidates)
frequent_bridges = [node for node, count in bridge_count.items() if count >= 3]
bridge_targets = nodes[
    nodes["Id"].isin(frequent_bridges) &
    ~nodes["type"].isin(["policy", "private_policy", "citizen"])
]["Id"].tolist()
nodes.loc[nodes["Id"].isin(bridge_targets), "type"] = "bridge"

# 8. Save the typed node table and filtered edge list
nodes.drop(columns=["normalized_label"], inplace=True)
nodes.to_csv("nodes_with_type.csv", index=False)
edges.to_csv("edges_filtered.csv", index=False)
